# SoftVertexTriangleStitch – StableNeoHookean symbol derivation

Each (vertex, triangle) pair forms a tetrahedron: 1 vertex from mesh A and 3 vertices of a triangle from mesh B.
We use the same **StableNeoHookean** formulation as in simplified_stable_neo_hookean_3d for shape-keeping energy (may be changed later; formula repeated here for self-contained reference).
- Invariants: I2(VecF) = VecF·VecF, I3(F) = det(F)
- Energy: E = (mu/2*(I2-3) - mu*(I3-1) + lambda/2*(I3-1)^2) + (mu/lambda)^2
- Output: E, dEdVecF, ddEddVecF (same signatures as stable_neo_hookean_3d) for use in inter_primitive backend.

In [1]:
import sys
import os
sys.path.append('../')
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")

def I2(VecF):
    return VecF.dot(VecF)
    
def I3(F):
    return F.det()

ModuleNotFoundError: No module named 'sympy'

In [ ]:
mu, lmd = symbols("mu lambda")
F = Eigen.Matrix("F", 3, 3)
F

In [ ]:
VecF = F.Vectorize("VecF")
VecF

In [ ]:
I2 = I2(VecF)
I2

In [ ]:
I3 = I3(F)
I3

In [ ]:
E = (mu/2*(I2-3) - mu * (I3-1) + lmd/2*(I3-1)**2) + (mu/lmd)**2
E

In [ ]:
dEdVecF = VecDiff(E, VecF)
dEdVecF

In [ ]:
ddEddVecF = VecDiff(dEdVecF, VecF)
ddEddVecF

In [ ]:
Cl = Gen.Closure(mu, lmd, VecF)

In [ ]:
s = f'''
{Cl("E",E)}
{Cl("dEdVecF",dEdVecF)}
{Cl("ddEddVecF",ddEddVecF)}
'''
print(s)

f = open( backend_source_dir('cuda') / 'inter_primitive_effect_system/constitutions/details/soft_vertex_triangle_stitch.inl', 'w')
f.write(s)
f.close()